### Animationen mit asyncio

Folgendes Pattern erstellt eine einfache Animation mit `asyncio`.
Eine solche Animation läuft im Hintergrund und blockiert das Notebook nicht.

```python
import asyncio

is_running = True

def animation(canvas, fps=25):
    async def animate():
        while is_running:
            with hold_canvas(canvas):
                # draw frame
            await asyncio.sleep(1/fps)

    task = asyncio.create_task(animate(), name='my_animation')
    return task
```


Der Aufruf von `animate()` gibt eine sog. Coroutine zurückzugeben 
(ähnlich wie eine Generator-Funktion ein Generator  zurück gibt).
Aus dieser Coroutine macht dann `asyncio.create_task(animate())` einen Task,
der in den Eventloop von Jupyterlab aufgenommen und gestartet wird.


Jedesmal, wenn `await asyncio.sleep(t)` angetroffen wird,
wird der Task pausiert. Der Eventloop kümmert sich um die anderen Tasks.
Nach t Sekunden, wird der Task wieder fortgesetzt.

```
draw frame 0
     ↓
await asyncio.sleep(1/fps)
     ↓
Eventloop kümmert sich um andere Aufgaben
     ↓
resume (nach 1/fps sec)
     ↓
draw frame 1
     ↓
...
```

Es ist wichtig, `task` zurückzugeben:
- Fehlermeldungen von `animate` werden nicht angezeigt.  
  `task.exception()` zeigt diese an.
- `task.cancel()` erlaubt, die Animation zu stoppen.
- ` task.get_name()` liefert Name des Tasks.
- `task.done()` ist True, falls der Task beendet ist (erfolgreich beendet, Fehler geworfen oder cancelled).
- `task.result()` liefert den Rückgabewert der Coroutine, falls Task erfolgreich beendet.
  Wirf Fehler, falls Task noch nicht beendet, cancelled oder falls Task Fehler verursachte.

**Laufende Tasks anzeigen, finden und canceln**:   
`asyncio.all_tasks()` liefert eine Menge mit allen laufenden Tasks.

```python
def show_tasks():
    for task in asyncio.all_tasks():
        print(f'Task: {task.get_name()}')
       
def cancel_task(task_name):
    '''cancel task mit Namen name'''
    for task in asyncio.all_tasks():
        if task.get_name() == task_name:
            print(f'trying to cancel task {task_name}')
            return task.cancel()
```

In [ ]:
import asyncio


count = 0


def step():
    '''ein Animationsschritt'''
    global count
    count += 1
    # countx += 1


def run():
    '''erhoeht count[0] jede Sekunde waehrend 60 Sekunden'''
    async def my_animation():
        for i in range(60):
            step()
            await asyncio.sleep(1)
        return i

    task = asyncio.create_task(my_animation(), name='my_animation')
    return task

In [ ]:
task = run()

In [ ]:
for task in asyncio.all_tasks():
    print(f'Task: {task.get_name()}')

print('count:', count)
print('name:', task.get_name())
print('done:', task.done())
if task.done():
    print('result:', task.result())

### Tickende Uhr

In [ ]:
import widget_helpers as W
import clock_helpers
from IPython.display import display
from ipycanvas import hold_canvas


class Clock:
    def __init__(self):
        self.canvas = W.get_canvas()
        self.is_running = False
        self.center = (50, 50)
        self.radius = 40

        display(self.canvas)

    def start(self):
        self.is_running = True
        self._run()

    def stop(self):
        self.is_running = False

    def step(self):
        with hold_canvas(self.canvas):
            self.canvas.clear()
            clock_helpers.draw_clockface(self.canvas, self.center, self.radius)
            clock_helpers.draw_hands(self.canvas, self.center, self.radius)

    def _run(self):
        async def animate():
            while self.is_running:
                self.step()
                await asyncio.sleep(1)

        self.task = asyncio.create_task(animate(), name='ticking_clock')

In [ ]:
clock = Clock()

In [ ]:
clock.start()

In [ ]:
clock.stop()

In [ ]:
for task in asyncio.all_tasks():
    print(f'Task: {task.get_name()}')